# Coin Sorter — Colab Training Notebook

Trains a YOLO-cls classifier on coin images captured on the Pi, then exports
to ONNX for inference back on the Pi.

**Before running:**
1. On the Pi, run `scripts/zip_dataset.sh` to bundle `data/raw/` into `data.zip`.
2. Upload `data.zip` to `MyDrive/coin_sorter/data.zip` on Google Drive.
3. Runtime → Change runtime type → T4 GPU.
4. Runtime → Run all.

The notebook is idempotent — safe to run all cells multiple times in the same session.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

## 2. Install dependencies

In [ ]:
!pip install -q ultralytics onnx onnxruntime

## 3. Unzip dataset (idempotent)

Re-extracts only if the destination is empty or older than the zip.

In [ ]:
import os, shutil, zipfile, pathlib

DRIVE_BASE = '/content/drive/MyDrive/coin_sorter'
DATA_ZIP   = f'{DRIVE_BASE}/data.zip'
MODEL_DIR  = f'{DRIVE_BASE}/models'
DATA_ROOT  = '/content/data'

assert os.path.exists(DATA_ZIP), f'Missing {DATA_ZIP} — upload data.zip first'
os.makedirs(MODEL_DIR, exist_ok=True)

needs_extract = not os.path.isdir(DATA_ROOT) or not os.listdir(DATA_ROOT)
if not needs_extract:
    needs_extract = os.path.getmtime(DATA_ZIP) > os.path.getmtime(DATA_ROOT)

if needs_extract:
    if os.path.isdir(DATA_ROOT):
        shutil.rmtree(DATA_ROOT)
    os.makedirs(DATA_ROOT, exist_ok=True)
    with zipfile.ZipFile(DATA_ZIP) as z:
        z.extractall(DATA_ROOT)
    print('Extracted to', DATA_ROOT)
else:
    print('Dataset already present at', DATA_ROOT)

# Detect the actual class-folders root. zip_dataset.sh produces data.zip with
# class folders at the top level; if you uploaded a zip that contains a
# wrapping folder we re-point at it.
entries = [p for p in pathlib.Path(DATA_ROOT).iterdir() if p.is_dir()]
if len(entries) == 1 and not any((entries[0] / sub).is_dir() for sub in ('train', 'val')):
    inner_children = [p for p in entries[0].iterdir() if p.is_dir()]
    if inner_children:
        DATA_ROOT = str(entries[0])
print('Class folders:', sorted(p.name for p in pathlib.Path(DATA_ROOT).iterdir() if p.is_dir()))

## 4. Train

Ultralytics' classification training expects a folder-per-class layout and
will auto-split into train/val if you don't pre-split. To make the split
deterministic, re-run this cell with the same `seed`.

In [ ]:
from ultralytics import YOLO

BASE_MODEL = 'yolo11n-cls.pt'   # swap for 'yolov8n-cls.pt' to A/B
EPOCHS     = 50
BATCH      = 64
IMGSZ      = 224
PROJECT    = 'runs'
NAME       = 'coin_classifier'

model = YOLO(BASE_MODEL)
results = model.train(
    data=DATA_ROOT,
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=IMGSZ,
    project=PROJECT,
    name=NAME,
    exist_ok=True,         # idempotent — overwrite the previous run dir
    patience=15,
    plots=True,
    seed=1337,
    # Augmentation knobs — full rotation matters most for coins.
    degrees=180,
    fliplr=0.5,
    hsv_h=0.02,
    hsv_s=0.30,
    hsv_v=0.30,
    scale=0.10,
    translate=0.05,
)

## 5. Validate + confusion matrix

Ultralytics already wrote a confusion matrix to the run directory. Display it
inline plus print top-1 / top-5 accuracy from validation.

In [ ]:
from IPython.display import Image, display
import glob, os

run_dir = sorted(glob.glob(f'{PROJECT}/{NAME}*'))[-1]
print('Run dir:', run_dir)

metrics = model.val(data=DATA_ROOT, imgsz=IMGSZ, batch=BATCH)
print(f'top1 = {metrics.top1:.4f}')
print(f'top5 = {metrics.top5:.4f}')

for fname in ('confusion_matrix.png', 'confusion_matrix_normalized.png', 'results.png'):
    path = os.path.join(run_dir, fname)
    if os.path.exists(path):
        display(Image(path))

## 6. Export to ONNX

`dynamic=False` produces a fixed-shape graph — smaller, faster on the Pi,
and matches the static input size we use in `infer.py`. Match `imgsz` to
the training value exactly.

In [ ]:
onnx_path = model.export(format='onnx', dynamic=False, imgsz=IMGSZ, opset=17, simplify=True)
print('Exported:', onnx_path)

## 7. Copy ONNX to Drive

In [ ]:
import shutil, datetime, pathlib

src = pathlib.Path(onnx_path)
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
dest_stable = pathlib.Path(MODEL_DIR) / 'coin_classifier.onnx'
dest_stamped = pathlib.Path(MODEL_DIR) / f'coin_classifier_{stamp}.onnx'
shutil.copy2(src, dest_stable)
shutil.copy2(src, dest_stamped)

print('Stable :', dest_stable)
print('Stamped:', dest_stamped)
print('\nOn the Pi, run:\n  ./scripts/fetch_model.sh', dest_stable)